### Step 1: Mount Google Drive
We will mount your Google Drive to ensure that all transcripts are saved securely and won't be lost when the Colab session ends.

In [ ]:
from google.colab import drive
import os
import json

# Mount Google Drive
drive.mount('/content/drive')

# Create a dedicated directory for your transcripts
transcript_dir = '/content/drive/MyDrive/Whisper_Transcripts'
os.makedirs(transcript_dir, exist_ok=True)
os.makedirs('/content/incoming', exist_ok=True)

# Path to the deduplication registry
registry_path = os.path.join(transcript_dir, 'transcripts_registry.json')

def load_registry():
    if os.path.exists(registry_path):
        with open(registry_path, 'r', encoding='utf-8') as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return {}
    return {}

def save_registry(registry):
    with open(registry_path, 'w', encoding='utf-8') as f:
        json.dump(registry, f, indent=4)

print(f"Transcripts directory: {transcript_dir}")
print(f"Registry file: {registry_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Transcripts directory: /content/drive/MyDrive/Whisper_Transcripts
Registry file: /content/drive/MyDrive/Whisper_Transcripts/transcripts_registry.json


### Step 2: Install faster-whisper
We will install the `faster-whisper` library which utilizes CTranslate2 for high-performance Whisper model inference.

In [ ]:
!pip install faster-whisper "av<19"


In [ ]:
# Configure LD_LIBRARY_PATH and explicitly pre-load CUDA 12 shared libraries to memory
import os
import sys
import ctypes

colab_dist_packages = '/usr/local/lib/python3.13/dist-packages'
cublas_lib_dir = os.path.join(colab_dist_packages, 'nvidia', 'cublas', 'lib')
cudnn_lib_dir = os.path.join(colab_dist_packages, 'nvidia', 'cudnn', 'lib')

# Prepend directories to LD_LIBRARY_PATH
current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
new_paths = [cublas_lib_dir, cudnn_lib_dir]
existing_paths = [p for p in new_paths if os.path.exists(p)]

if existing_paths:
    os.environ['LD_LIBRARY_PATH'] = ":".join(existing_paths) + ":" + current_ld_path
    print("Successfully configured LD_LIBRARY_PATH for CUDA 12.")

# Force load the dynamic libraries directly into memory to bypass search resolution errors
target_libs = [
    os.path.join(cublas_lib_dir, 'libcublas.so.12'),
    os.path.join(cublas_lib_dir, 'libcublasLt.so.12'),
    os.path.join(cudnn_lib_dir, 'libcudnn.so.9')
]

for lib_path in target_libs:
    if os.path.exists(lib_path):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
            print(f"  - Loaded: {os.path.basename(lib_path)}")
        except Exception as e:
            print(f"  - Failed to pre-load {os.path.basename(lib_path)}: {e}")
    else:
        print(f"  - Warning: {lib_path} not found on disk.")

Successfully configured LD_LIBRARY_PATH for CUDA 12.
  - Loaded: libcublas.so.12
  - Loaded: libcublasLt.so.12
  - Loaded: libcudnn.so.9


### Step 3: Initialize the Model
We will load the `large-v3` model (the most accurate Whisper model available in `faster-whisper`) running on GPU (float16) for fast processing.

In [ ]:
import torch
from faster_whisper import WhisperModel

# Re-determine device availability with the active CUDA process memory
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Re-initializing model on {device} using {compute_type}...")
# Instantiate the recommended high-performance large-v3-turbo model
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)

Re-initializing model on cuda using float16...


### Step 4: Chapter-Aware Transcription Pipeline

Replaces blind chunking with an architecture matched directly to chapter structure:
- Uses `ffprobe` chapter detection with sub-second dust-atom deduplication (`DUST_ATOM_MIN_S = 1.0s`).
- Silence-aware sub-chunking for chapters exceeding 1 hour (`MAX_CHUNK_SECONDS = 3600`), keeping RAM strictly under 1 GB without blind mid-word cuts.
- Robust duration detection (`get_audio_duration_robust`) resolving `.m4b`/`.m4a` metadata issues.
- Optimized VAD parameters preventing segment fragmentation (`min_silence_duration_ms: 1200`, `repetition_penalty: 1.1`).
- Non-recursive prompt conditioning (`condition_on_previous_text: False`) eliminating word-level timestamp token loops.
- Paragraph aggregation (`aggregate_to_paragraphs`): merges clause and word-level slices into mature ~25–35s paragraphs with 100% word timestamps preserved, matching Audiobench production standards (Goggins/Hanes).
- Full two-stage chapter merge so sub-chunk boundaries remain invisible in the exported Audiobench JSON schema.
- **Multi-Layer Integrity Guards**:
  - **Calibrated Timeline Coverage**: Enforces >= 95% duration coverage for long files (> 30m) and max tail gaps (<= max(120s, 3%)).
  - **Chapter Boundary Verification**: Confirms final authored chapter was reached (tail gap <= max(180s, 5% chapter duration)), accommodating legitimate quiet outros and meditation pauses without false positives.
  - **T0 -> T2 File Immutability Check**: Detects concurrent file writes or in-flight mutations during transcription, raising a distinct diagnostic.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PyAV 19+ Compatibility Patch
# PyAV 19 removed metadata_errors, but faster-whisper still passes it.
# ─────────────────────────────────────────────────────────────────────────────
import av
if hasattr(av, "open") and not hasattr(av, "_patched_for_metadata_errors"):
    _orig_av_open = av.open
    def _safe_av_open(*args, **kwargs):
        kwargs.pop("metadata_errors", None)
        return _orig_av_open(*args, **kwargs)
    av.open = _safe_av_open
    av._patched_for_metadata_errors = True

import subprocess
import os
import math
import json
import gc
import hashlib
import re
import torch
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime, timezone
from difflib import SequenceMatcher

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────
MAX_CHUNK_SECONDS = 3600    # safe ceiling per decode call (measured peak ~0.791 GB/h at 1.0 GB free)
SILENCE_WINDOW    = 120     # ±2 min search window around sub-split target, clamped to chapter bounds
DUST_ATOM_MIN_S   = 1.0  # filter sub-second duplicate chapter atoms (e.g. 48 Laws pattern)
CHUNK_DIR         = '/content/chunks'

WHISPER_PARAMS = dict(
    beam_size=5,
    word_timestamps=True,
    vad_filter=True,
    vad_parameters={
        "min_silence_duration_ms": 1200,
        "speech_pad_ms": 300,
        "min_speech_duration_ms": 500,
    },
    condition_on_previous_text=False,
    repetition_penalty=1.1,
    no_speech_threshold=0.6,
    log_prob_threshold=-1.0,
    compression_ratio_threshold=2.4,
    temperature=0,
)


# ─────────────────────────────────────────────────────────────────────────────
# Data class
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class ChapterInfo:
    chapter_index: Optional[int]   # None for chapterless files
    title: str
    start_s: float
    end_s: float

    @property
    def duration(self):
        return self.end_s - self.start_s


# ─────────────────────────────────────────────────────────────────────────────
# 1. Duration detection (fixes format=duration N/A bug on .m4b/.m4a)
# ─────────────────────────────────────────────────────────────────────────────
def get_audio_duration_robust(path):
    """Get audio duration via stream header first, then format, then decode-pass.
    Never returns 0.0 for a valid audio file."""
    def _probe(entries):
        r = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', entries,
             '-of', 'default=noprint_wrappers=1:nokey=1', path],
            capture_output=True, text=True, timeout=15,
        )
        try:
            return float(r.stdout.strip())
        except (ValueError, TypeError):
            return None

    # Stream duration: works on .m4b/.m4a where format=duration returns N/A
    d = _probe('stream=duration')
    if d and d > 0:
        return d

    # Format duration: works on plain mp3/wav/flac
    d = _probe('format=duration')
    if d and d > 0:
        return d

    # Decode-pass fallback: reads the whole file, always accurate
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-f', 'null', '-', '-i', path],
        capture_output=True, text=True, timeout=1800,
    )
    for line in reversed(r.stderr.splitlines()):
        m = re.search(r'Duration:\s*(\d+):(\d+):(\d+\.?\d*)', line)
        if m:
            h, mn, s = m.groups()
            return int(h) * 3600 + int(mn) * 60 + float(s)

    raise RuntimeError(f"Could not determine duration for: {path}")


# ─────────────────────────────────────────────────────────────────────────────
# 2. Chapter detection with dust-atom deduplication
# ─────────────────────────────────────────────────────────────────────────────
def detect_chapters(path):
    """Detect authored chapters via ffprobe.

    Filters dust atoms (e.g. 48 Laws of Power has ~0.026s duplicate chapter atoms
    before each real chapter). Assigns a fresh 0-based chapter_index from the
    filtered list position — never trusts the container's raw chapter id field.

    Returns:
        List of ChapterInfo. If no real chapters found, returns a single
        ChapterInfo(chapter_index=None, title='Full Recording', ...) sentinel.
        Callers must treat chapter_index=None as chapterless — segments get
        chapter_id: null in the final JSON, matching production DB behaviour.
    """
    r = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_chapters', path],
        capture_output=True, text=True, timeout=15,
    )
    try:
        raw = json.loads(r.stdout).get('chapters', [])
    except json.JSONDecodeError:
        raw = []

    # Filter dust atoms — sub-DUST_ATOM_MIN_S duplicate atoms from certain encoders
    real = [
        ch for ch in raw
        if float(ch['end_time']) - float(ch['start_time']) >= DUST_ATOM_MIN_S
    ]

    if not real:
        duration = get_audio_duration_robust(path)
        return [ChapterInfo(
            chapter_index=None,
            title='Full Recording',
            start_s=0.0,
            end_s=duration,
        )]

    return [
        ChapterInfo(
            chapter_index=i,        # 0-based, gapless, assigned from filtered position
            title=ch.get('tags', {}).get('title', f'Chapter {i+1}').strip() or f'Chapter {i+1}',
            start_s=float(ch['start_time']),
            end_s=float(ch['end_time']),
        )
        for i, ch in enumerate(real)
    ]


# ─────────────────────────────────────────────────────────────────────────────
# 3. Silence-aware sub-split boundary finder
# ─────────────────────────────────────────────────────────────────────────────
def find_silence_near(path, chapter_start_s, chapter_end_s, target_s,
                      window_s=SILENCE_WINDOW, noise_db=-35, min_silence_dur=0.5):
    """Find the silence boundary nearest to target_s within the chapter's bounds.

    Search window is clamped to [chapter_start_s, chapter_end_s] — structurally
    cannot cross into adjacent chapter audio even when chapter is only slightly
    over MAX_CHUNK_SECONDS (e.g. 3650s chapter, target at 3600s: unclamped window
    would reach 3720s, clamped to 3650s).

    Returns:
        (best_t, is_fallback):
          is_fallback=False  → silence found → use direct extend() join
          is_fallback=True   → no silence in window → arithmetic cut → LCS stitch
    """
    # Clamped search window — never crosses chapter boundary
    search_start = max(chapter_start_s, target_s - window_s)
    search_end   = min(chapter_end_s,   target_s + window_s)
    seek         = search_start
    dur          = search_end - search_start

    if dur <= 0:
        return target_s, True  # degenerate window: fallback

    r = subprocess.run([
        'ffmpeg', '-ss', str(seek), '-t', str(dur),
        '-i', path,
        '-af', f'silencedetect=noise={noise_db}dB:duration={min_silence_dur}',
        '-f', 'null', '-',
    ], capture_output=True, text=True, timeout=120)

    best_t = None
    for line in r.stderr.splitlines():
        if 'silence_start' in line:
            try:
                # ffmpeg reports silence_start relative to the -ss seek point
                t_relative = float(line.split('silence_start:')[1].strip())
                t_absolute = seek + t_relative
                if chapter_start_s <= t_absolute <= chapter_end_s:  # safety check
                    if best_t is None or abs(t_absolute - target_s) < abs(best_t - target_s):
                        best_t = t_absolute
            except (ValueError, IndexError):
                continue

    if best_t is None:
        return target_s, True   # fallback: arithmetic cut, LCS stitch at this seam
    return best_t, False        # clean silence cut, direct extend()


# ─────────────────────────────────────────────────────────────────────────────
# 4. WAV chunk extractor — writes to disk, Python never buffers PCM
# ─────────────────────────────────────────────────────────────────────────────
def extract_chunk_to_wav(path, start_s, end_s, out_wav):
    """Extract a time window to a 16kHz mono WAV using ffmpeg.

    Python never holds the decoded PCM in memory — ffmpeg writes directly to disk.
    Peak RAM during this call: negligible subprocess overhead only.
    The returned WAV path is passed to model.transcribe(); faster-whisper's
    internal decode_audio() will buffer it (peak ~6 bytes/sample), but only
    for the chunk duration, not the full file.
    """
    os.makedirs(os.path.dirname(out_wav), exist_ok=True)
    subprocess.run([
        'ffmpeg', '-y',
        '-ss', str(start_s), '-to', str(end_s),
        '-i', path,
        '-vn', '-ar', '16000', '-ac', '1', '-acodec', 'pcm_s16le',
        out_wav,
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return out_wav


# ─────────────────────────────────────────────────────────────────────────────
# 5. LCS stitcher — used ONLY at arithmetic-fallback seams
# ─────────────────────────────────────────────────────────────────────────────
def _lcs_stitch(prev_segs, next_segs, overlap_budget_s=90):
    """Merge two segment lists at an arithmetic-fallback seam using LCS.

    Only invoked when find_silence_near() returned is_fallback=True — i.e.
    when no silence was found and a blind arithmetic cut was made.
    Clean chapter boundaries and clean silence cuts use direct extend() instead.

    Strategy: find the overlap region by timestamp, use SequenceMatcher on
    word text to find the best join point, discard duplicated words.
    Falls back to midpoint cut if LCS match quality is too low.
    """
    if not prev_segs or not next_segs:
        prev_segs.extend(next_segs)
        return prev_segs

    # Find overlap region: segments from prev_segs whose end > next_segs[0].start - budget
    overlap_start = next_segs[0]['start'] - overlap_budget_s
    prev_overlap = [s for s in prev_segs if s['end'] >= overlap_start]
    prev_clean   = [s for s in prev_segs if s['end'] < overlap_start]

    prev_words = ' '.join(s['text'] for s in prev_overlap).split()
    next_words = ' '.join(s['text'] for s in next_segs[:10]).split()

    matcher = SequenceMatcher(None, prev_words[-60:], next_words[:60], autojunk=False)
    best = matcher.find_longest_match(0, len(prev_words[-60:]), 0, len(next_words[:60]))

    if best.size >= 3:
        # Good LCS match: discard prev_overlap's trailing words past the match point
        # and use next_segs from the matching position onward
        join_idx_in_next = best.b + best.size
        # Find which next segment contains word index join_idx_in_next
        word_count = 0
        keep_from  = 0
        for i, seg in enumerate(next_segs):
            wc = len(seg['text'].split())
            if word_count + wc >= join_idx_in_next:
                keep_from = i
                break
            word_count += wc
        return list(prev_clean) + list(prev_overlap) + list(next_segs[keep_from:])
    else:
        # Low match quality: midpoint cut — discard next_segs that overlap prev_segs[-1].end
        cutoff = prev_segs[-1]['end']
        return list(prev_segs) + [s for s in next_segs if s['start'] >= cutoff - 0.1]


# ─────────────────────────────────────────────────────────────────────────────
# 6. Helper: build the sub-chunk plan for one chapter
# ─────────────────────────────────────────────────────────────────────────────
def _plan_sub_chunks(path, chapter):
    """Return list of (start_s, end_s, is_fallback_seam_before) for one chapter.

    Chapters within MAX_CHUNK_SECONDS get a single sub-chunk with no split.
    Longer chapters are silence-split into ≤MAX_CHUNK_SECONDS pieces.
    is_fallback_seam_before=True on a sub-chunk means the join at its START
    should use LCS stitching, not direct extend().
    """
    if chapter.duration <= MAX_CHUNK_SECONDS:
        return [(chapter.start_s, chapter.end_s, False)]

    sub_chunks = []
    seg_start  = chapter.start_s
    target     = chapter.start_s + MAX_CHUNK_SECONDS
    first      = True

    while seg_start < chapter.end_s - 30:  # don't create a tiny tail sliver < 30s
        remaining = chapter.end_s - seg_start
        if remaining <= MAX_CHUNK_SECONDS + SILENCE_WINDOW:
            # Last sub-chunk: take everything to chapter end
            sub_chunks.append((seg_start, chapter.end_s, False if first else _prev_fallback))
            break

        best_t, is_fallback = find_silence_near(
            path,
            chapter_start_s=chapter.start_s,
            chapter_end_s=chapter.end_s,
            target_s=target,
        )
        sub_chunks.append((seg_start, best_t, False if first else _prev_fallback))
        _prev_fallback = is_fallback
        seg_start = best_t
        target    = best_t + MAX_CHUNK_SECONDS
        first     = False

    return sub_chunks


# ─────────────────────────────────────────────────────────────────────────────
# 7. SHA-256 hash (preserved from existing notebook)
# ─────────────────────────────────────────────────────────────────────────────
def calculate_sha256(file_path, chunk_size=8192):
    sha256 = hashlib.sha256()
    with open(file_path, 'rb') as f:
        while True:
            data = f.read(chunk_size)
            if not data:
                break
            sha256.update(data)
    return sha256.hexdigest()


def format_seconds_to_minutes(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# 7.5. Paragraph aggregator — produces mature ~25-35s paragraphs matching Goggins/Hanes
# ─────────────────────────────────────────────────────────────────────────────
def aggregate_to_paragraphs(segments, min_duration=26.0, max_duration=38.0, pause_threshold=0.8):
    """Aggregate raw micro-segments into cohesive ~28–32s paragraphs.

    Preserves 100% of word timestamps in each segment's words array while grouping
    speech into narrative units that match Audiobench production standards
    (David Goggins / Cameron Hanes ~30s paragraph structure).

    A paragraph split is triggered when:
      1. Accumulated duration >= min_duration AND text ends with sentence-ending punctuation (. ! ? " ” '), OR
      2. Accumulated duration >= min_duration AND an acoustic pause >= pause_threshold is detected, OR
      3. Accumulated duration >= max_duration (hard ceiling preventing run-on blocks).
    """
    if not segments:
        return []

    paragraphs = []
    curr_words = []
    curr_texts = []
    curr_start = None
    curr_end = None
    curr_chap = None
    curr_durs = []
    curr_avg_logprob = []
    curr_no_speech = []

    for s in segments:
        s_start = s['start']
        s_end = s['end']
        s_text = s['text'].strip()
        s_words = s.get('words', [])
        s_chap = s.get('chapter_id')
        dur_s = s_end - s_start
        avg_lp = s.get('avg_logprob', -0.5)
        ns_prob = s.get('no_speech_prob', 0.0)

        if not s_text:
            continue

        if curr_start is None:
            curr_start = s_start
            curr_end = s_end
            curr_chap = s_chap
            curr_texts = [s_text]
            curr_words = list(s_words)
            curr_durs = [dur_s]
            curr_avg_logprob = [avg_lp]
            curr_no_speech = [ns_prob]
            continue

        dur = s_end - curr_start
        pause = s_start - curr_end
        last_text = curr_texts[-1] if curr_texts else ''
        ends_sentence = last_text[-1:] in ('.', '!', '?', '"', '”', "'")

        should_split = (
            (s_chap != curr_chap) or
            (dur >= min_duration and ends_sentence) or
            (dur >= min_duration and pause >= pause_threshold) or
            (dur >= max_duration)
        )

        if should_split:
            full_text = ' '.join(curr_texts).strip()
            tot_d = curr_end - curr_start
            if tot_d > 0 and curr_durs:
                weighted_lp = sum(d * lp for d, lp in zip(curr_durs, curr_avg_logprob)) / tot_d
                weighted_ns = sum(d * ns for d, ns in zip(curr_durs, curr_no_speech)) / tot_d
            else:
                weighted_lp = sum(curr_avg_logprob) / len(curr_avg_logprob) if curr_avg_logprob else -0.5
                weighted_ns = sum(curr_no_speech) / len(curr_no_speech) if curr_no_speech else 0.0

            paragraphs.append({
                "text":           full_text,
                "start":          round(curr_start, 3),
                "end":            round(curr_end, 3),
                "speaker":        None,
                "avg_logprob":    round(weighted_lp, 4),
                "no_speech_prob": round(weighted_ns, 4),
                "chapter_id":     curr_chap,
                "duration":       round(tot_d, 3),
                "word_count":     len(full_text.split()),
                "words":          curr_words,
            })
            curr_start = s_start
            curr_end = s_end
            curr_chap = s_chap
            curr_texts = [s_text]
            curr_words = list(s_words)
            curr_durs = [dur_s]
            curr_avg_logprob = [avg_lp]
            curr_no_speech = [ns_prob]
        else:
            curr_texts.append(s_text)
            curr_words.extend(s_words)
            curr_end = s_end
            curr_durs.append(dur_s)
            curr_avg_logprob.append(avg_lp)
            curr_no_speech.append(ns_prob)

    if curr_texts:
        full_text = ' '.join(curr_texts).strip()
        tot_d = curr_end - curr_start
        if tot_d > 0 and curr_durs:
            weighted_lp = sum(d * lp for d, lp in zip(curr_durs, curr_avg_logprob)) / tot_d
            weighted_ns = sum(d * ns for d, ns in zip(curr_durs, curr_no_speech)) / tot_d
        else:
            weighted_lp = sum(curr_avg_logprob) / len(curr_avg_logprob) if curr_avg_logprob else -0.5
            weighted_ns = sum(curr_no_speech) / len(curr_no_speech) if curr_no_speech else 0.0

        paragraphs.append({
            "text":           full_text,
            "start":          round(curr_start, 3),
            "end":            round(curr_end, 3),
            "speaker":        None,
            "avg_logprob":    round(weighted_lp, 4),
            "no_speech_prob": round(weighted_ns, 4),
            "chapter_id":     curr_chap,
            "duration":       round(tot_d, 3),
            "word_count":     len(full_text.split()),
            "words":          curr_words,
        })

    return paragraphs

# 8. Main entry point
# ─────────────────────────────────────────────────────────────────────────────
def transcribe_chapter_pipeline(file_path, language=None, stream_segments=False):
    """Transcribe any audio file via chapter-aware, RAM-safe pipeline.

    Architecture:
      Stage 0 — Plan: detect chapters, plan sub-chunks for large chapters
      Stage 1 — Transcribe: one WAV per sub-chunk, merge back to chapter
      Stage 2 — Output: assign chapter_id, write JSON + TXT, update registry

    Sub-chunk splits are a transcription-time RAM-safety detail only.
    They are invisible in the output: only real authored chapters appear in
    the JSON chapters[] array; segments carry the parent chapter's chapter_index.
    Chapterless files: chapters=[], all segments chapter_id=null.

    This matches production DB schema exactly:
      - Chaptered (Goggins): 1624/1624 segments have chapter_id set
      - Chapterless (The Truth Part 1): 824/824 segments have chapter_id=None

    Five Pipeline Integrity Guards:
      1. T0 Pre-Flight Check: post_hash_size == initial_size (fails fast if file grew during hashing before inference)
      2. T2 Post-Run Immutability: final_size == initial_size (catches concurrent writes/uploads mid-transcription)
      3. Timeline Coverage: >= 95.0% (>30m), >= 90.0% (5-30m), >= 80.0% (<=5m) (catches severely truncated output)
      4. Maximum Tail Gap: <= max(120.0s, 0.03 * duration) (permits natural outros while bounding multi-hour tail to 3%)
      5. Chapter Boundary Check: <= max(180.0s, 0.05 * last_chapter.duration) (confirms final authored chapter was reached)
    """
    file_path  = os.path.abspath(file_path)
    initial_size = os.path.getsize(file_path)
    initial_mtime = os.path.getmtime(file_path)
    file_hash  = calculate_sha256(file_path)

    # Immediate pre-flight check: if file grew during hashing, fail early before burning GPU time
    post_hash_size = os.path.getsize(file_path)
    assert post_hash_size == initial_size, (
        f"[PIPELINE INTEGRITY BREACH: T0 PRE-FLIGHT MUTATION] "
        f"File size changed from {initial_size:,} to {post_hash_size:,} bytes during SHA-256 calculation! "
        f"The file is actively growing. Aborting before starting Whisper inference."
    )
    base_name  = os.path.splitext(os.path.basename(file_path))[0]
    out_json   = os.path.join(transcript_dir, f"{base_name}.json")
    out_txt    = os.path.join(transcript_dir, f"{base_name}.txt")

    # Deduplication check (preserved from original notebook)
    registry = load_registry()
    if file_hash in registry:
        print(f"Skipping '{base_name}': already transcribed as '{registry[file_hash]}'")
        return

    print(f"\n{'='*60}")
    print(f"Transcribing: {os.path.basename(file_path)}")
    print(f"{'='*60}")

    # ── Stage 0: Plan ────────────────────────────────────────────────────────
    duration = get_audio_duration_robust(file_path)
    chapters = detect_chapters(file_path)

    is_chapterless = (len(chapters) == 1 and chapters[0].chapter_index is None)
    print(f"Duration: {format_seconds_to_minutes(duration)}")
    if is_chapterless:
        print(f"Chapters: none — single-pass with sub-splitting if needed")
    else:
        print(f"Chapters: {len(chapters)} authored chapters")

    os.makedirs(CHUNK_DIR, exist_ok=True)

    # ── Stage 1: Transcribe + sub-chunk merge ─────────────────────────────────
    all_segments      = []
    detected_language = language
    language_prob     = 1.0
    first_chapter     = True

    for chapter in chapters:
        sub_chunks = _plan_sub_chunks(file_path, chapter)
        n_subs     = len(sub_chunks)
        label      = 'Full Recording' if is_chapterless else chapter.title
        if n_subs > 1:
            print(f"\n  Chapter: {label!r}  ({format_seconds_to_minutes(chapter.duration)}) "
                  f"→ {n_subs} sub-chunks")
        else:
            print(f"\n  Chapter: {label!r}  ({format_seconds_to_minutes(chapter.duration)})")

        chapter_segments = []
        prev_was_fallback = False  # seam type BEFORE current sub-chunk

        for sub_idx, (sub_start, sub_end, seam_before_is_fallback) in enumerate(sub_chunks):
            wav_path = os.path.join(
                CHUNK_DIR,
                f"{base_name}_ch{chapter.chapter_index or 0}_sub{sub_idx:03d}.wav",
            )
            print(f"    Sub-chunk {sub_idx+1}/{n_subs}: "
                  f"{format_seconds_to_minutes(sub_start)}–{format_seconds_to_minutes(sub_end)}", end=' ')

            extract_chunk_to_wav(file_path, sub_start, sub_end, wav_path)

            params = dict(WHISPER_PARAMS)
            if language:
                params['language'] = language

            raw_gen, info = model.transcribe(wav_path, **params)

            if first_chapter and sub_idx == 0:
                detected_language = info.language
                language_prob     = info.language_probability
                first_chapter     = False

            raw_segs = []
            for seg in raw_gen:
                # Offset timestamps to file-absolute time
                abs_start = round(seg.start + sub_start, 3)
                abs_end   = round(seg.end   + sub_start, 3)

                if stream_segments:
                    print(f"\n      [{format_seconds_to_minutes(abs_start)}→"
                          f"{format_seconds_to_minutes(abs_end)}] {seg.text.strip()}")

                words_list = []
                if seg.words:
                    for w in seg.words:
                        w_s = round(w.start + sub_start, 3)
                        w_e = round(w.end   + sub_start, 3)
                        words_list.append({
                            "word":        w.word.strip(),
                            "start":       w_s,
                            "end":         w_e,
                            "probability": round(w.probability, 4),
                            "speaker":     None,
                            "duration":    round(w_e - w_s, 3),
                            "midpoint":    round((w_s + w_e) / 2, 3),
                        })

                raw_segs.append({
                    "text":         seg.text.strip(),
                    "start":        abs_start,
                    "end":          abs_end,
                    "speaker":      None,
                    "avg_logprob":  round(seg.avg_logprob, 4),
                    "no_speech_prob": round(seg.no_speech_prob, 4),
                    "chapter_id":   chapter.chapter_index,  # None for chapterless
                    "duration":     round(abs_end - abs_start, 3),
                    "word_count":   len(seg.text.strip().split()),
                    "words":        words_list,
                })

            print(f"→ {len(raw_segs)} segments")

            # Merge sub-chunks: LCS only at arithmetic-fallback seams
            if not chapter_segments:
                chapter_segments = raw_segs
            elif seam_before_is_fallback:
                # Arithmetic cut was used before this sub-chunk — LCS stitch
                chapter_segments = _lcs_stitch(chapter_segments, raw_segs)
            else:
                # Clean silence cut — direct extend, no stitcher needed
                chapter_segments.extend(raw_segs)

            # Cleanup: delete WAV immediately, force GC
            try:
                os.remove(wav_path)
            except OSError:
                pass
            del raw_segs
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Sub-chunk merge complete. chapter_segments is now one continuous list.
        # Sub-chunk boundaries are gone — no sub-chunk labels anywhere.
        chapter_paragraphs = aggregate_to_paragraphs(chapter_segments)
        print(f"    Paragraph aggregation: {len(chapter_segments)} raw slices → {len(chapter_paragraphs)} mature paragraphs")
        all_segments.extend(chapter_paragraphs)

    # ── Stage 2: Assign global IDs, build JSON output ────────────────────────
    for idx, seg in enumerate(all_segments):
        seg['id'] = idx

    # Verify Stage 1 merge integrity and timeline coverage (Fail loudly on corrupted or truncated merge)
    assert len(all_segments) > 0, (
        f"Transcription failure: produced 0 segments for {os.path.basename(file_path)} "
        f"(duration: {format_seconds_to_minutes(duration)})"
    )

    assert all_segments[-1]['end'] > all_segments[0]['start'], (
        f"Merge failure: total timeline collapsed (start={all_segments[0]['start']}s, end={all_segments[-1]['end']}s)"
    )
    for i in range(1, len(all_segments)):
        assert all_segments[i]['start'] >= all_segments[i-1]['end'] - 0.5, (
            f"Merge failure at segment {i}: backward time jump detected "
            f"(seg {i} starts at {all_segments[i]['start']}s, prev ends at {all_segments[i-1]['end']}s)"
        )

    # ── Multi-Layer Truncation & Coverage Guards ─────────────────────────
    # 1. Total duration coverage percentage
    coverage_pct = (all_segments[-1]['end'] / duration * 100) if duration > 0 else 100.0
    min_required_coverage = 95.0 if duration > 1800 else (90.0 if duration > 300 else 80.0)
    assert coverage_pct >= min_required_coverage, (
        f"Merge failure: severely truncated output ({coverage_pct:.2f}% coverage of "
        f"{format_seconds_to_minutes(duration)}, minimum required {min_required_coverage:.1f}%). "
        f"Last transcribed segment ends at {format_seconds_to_minutes(all_segments[-1]['end'])}."
    )

    # 2. Maximum allowable tail gap (detects missing trailing audio while permitting natural outros)
    tail_gap = duration - all_segments[-1]['end']
    max_allowed_tail_gap = max(120.0, 0.03 * duration)
    assert tail_gap <= max_allowed_tail_gap, (
        f"Truncation detected: audio timeline extends to {format_seconds_to_minutes(duration)} "
        f"but last transcribed segment ends at {format_seconds_to_minutes(all_segments[-1]['end'])} "
        f"(missing tail gap: {tail_gap:.1f}s, maximum allowed: {max_allowed_tail_gap:.1f}s)."
    )

    # 3. Chapter boundary check: ensure the final authored chapter was actually reached.
    # Calibrated to max(180s, 5% chapter duration) to accommodate legitimate quiet outros,
    # meditation pauses, and loose publisher metadata without false positives.
    if not is_chapterless and chapters:
        last_chapter = chapters[-1]
        expected_end = min(last_chapter.end_s, duration)
        allowed_chapter_gap = max(180.0, 0.05 * last_chapter.duration)
        chapter_gap = expected_end - all_segments[-1]['end']
        assert chapter_gap <= allowed_chapter_gap, (
            f"Chapter truncation detected: final authored chapter '{last_chapter.title}' ends at "
            f"{format_seconds_to_minutes(expected_end)}, but transcription ends at "
            f"{format_seconds_to_minutes(all_segments[-1]['end'])} "
            f"(gap: {chapter_gap:.1f}s > {allowed_chapter_gap:.1f}s allowable)."
        )

    print(f"  [Merge Validation Passed] Verified {len(all_segments)} strictly monotonic segments spanning "
          f"{format_seconds_to_minutes(all_segments[-1]['end'])} ({coverage_pct:.2f}% coverage, tail gap {tail_gap:.1f}s).")

    # ── Guard: File Immutability (T0 -> T2 check) ────────────────────────────
    # If this fires, concurrent writes bypassed or survived staging isolation.
    final_size = os.path.getsize(file_path)
    assert final_size == initial_size, (
        f"[PIPELINE INTEGRITY BREACH: T0->T2 FILE MUTATION] "
        f"File size changed from {initial_size:,} to {final_size:,} bytes while Whisper was executing! "
        f"This indicates concurrent writes to '{file_path}' bypassed or survived staging promotion. "
        f"Atomic staging / settle guard did not isolate this file. Aborting transcript save to prevent corrupted database entry."
    )

    full_text = ' '.join(s['text'] for s in all_segments)

    # Build chapters array: only real authored chapters, never sub-chunk entries
    chapters_json = [] if is_chapterless else [
        {
            "chapter_index": ch.chapter_index,
            "title":         ch.title,
            "start_time":    round(ch.start_s, 3),
            "end_time":      round(ch.end_s, 3),
        }
        for ch in chapters
    ]

    # NOTE: Store initial_size (captured at T0 alongside file_hash and duration)
    # rather than re-evaluating os.path.getsize(file_path) at write time. This
    # guarantees that recorded size, hash, and duration remain 100% mutually consistent
    # even if the file on disk was modified during execution (preventing the Final Spin Part 04 bug).
    audiobench_data = {
        "schema_version":     "1",
        "audio": {
            "file_name":       os.path.basename(file_path),
            "file_size_bytes": initial_size,
            "format":          file_path.rsplit('.', 1)[-1].lower(),
            "duration_seconds": round(duration, 3),
            "sample_rate":     44100,
            "channels":        2,
            "file_hash":       file_hash,
        },
        "language":            detected_language,
        "language_probability": round(language_prob, 3),
        "duration_seconds":    round(duration, 3),
        "engine":              "whisper",
        "model_name":          "large-v3-turbo",
        "speaker_map":         {},
        "created_at":          datetime.now(timezone.utc).isoformat(),
        "chapters":            chapters_json,
        "is_cached":           False,
        "text":                full_text,
        "word_count":          len(full_text.split()),
        "segment_count":       len(all_segments),
        "segments":            all_segments,
    }

    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(audiobench_data, f, ensure_ascii=False, indent=2)

    with open(out_txt, 'w', encoding='utf-8') as f:
        for seg in all_segments:
            f.write(seg['text'] + '\n\n')

    registry[file_hash] = f"{base_name}.txt"
    save_registry(registry)

    avg_seg_dur = (sum(s["duration"] for s in all_segments) / len(all_segments)) if all_segments else 0.0
    avg_seg_words = (sum(s["word_count"] for s in all_segments) / len(all_segments)) if all_segments else 0.0
    print(f"\n✓ {len(all_segments)} paragraphs (avg {avg_seg_dur:.1f}s, {avg_seg_words:.0f} words) | "
          f"{len(chapters_json)} chapters | "
          f"{format_seconds_to_minutes(duration)}")
    print(f"  JSON → {out_json}")
    print(f"  TXT  → {out_txt}")


print("Chapter-aware transcription pipeline loaded.")
print(f"  MAX_CHUNK_SECONDS = {MAX_CHUNK_SECONDS}s ({MAX_CHUNK_SECONDS//3600}h)")
print(f"  DUST_ATOM_MIN_S   = {DUST_ATOM_MIN_S}s")
print(f"  VAD min_silence_duration_ms = {WHISPER_PARAMS['vad_parameters']['min_silence_duration_ms']}")


### Step 5: Batch Transcribe with Atomic Staging & Auto-Rescan

This pipeline processes queued media files safely and continuously without risk of transcribing partial or in-flight uploads:

#### Safe Upload Patterns
1. **Atomic Staging Directory (Recommended for Direct Colab Uploads)**:
   - When uploading files via Colab's left sidebar, upload into `/content/incoming/`.
   - While files are uploading over your browser connection, they remain in `/content/incoming/` and are ignored by the transcription worker.
   - Once uploads finish, the batch scanner automatically promotes fully settled files from `/content/incoming/` to `/content/` via POSIX atomic rename (`os.replace`), or you can run:
     ```bash
     !mv /content/incoming/* /content/
     ```
   - Because `mv` within the same filesystem is an atomic inode pointer swap, files appear in the queue instantaneously and 100% complete—never partially written.

2. **Google Drive Sync (Recommended for Large Batches)**:
   - Place files in Google Drive (`/content/drive/MyDrive/...`) and copy or process them directly.

#### Automated Safety Guards
- **Settle Guard**: Files modified < 15 seconds ago or actively changing in size are automatically held back until writes complete (protects both staged files and direct `/content` uploads).
- **T0 -> T2 Immutability Assertion**: If an audio file's byte count changes between the start and end of Whisper transcription, the transcript is rejected before saving with a distinct `[PIPELINE INTEGRITY BREACH]` diagnostic.
- **Dual-Layer Coverage & Chapter Boundary Guard**: Transcriptions with < 95% coverage, excessive tail gaps, or missing final chapters fail loudly rather than saving truncated output, with tolerances calibrated to avoid false rejections on natural quiet outros.
- **Continuous Auto-Rescan**: After completing a queue round, the script rescans for newly arrived files before packaging the final ZIP and triggering browser download.


In [ ]:
import os
import gc
import time
import subprocess
import glob
import zipfile
import torch
from google.colab import files

INCOMING_DIR = '/content/incoming'
os.makedirs(INCOMING_DIR, exist_ok=True)

MEDIA_EXTS = {
    '.mp3', '.m4b', '.m4a', '.wav', '.flac', '.aac', '.ogg',
    '.wma', '.opus', '.mp4', '.mkv', '.mov', '.webm', '.avi'
}

def is_playable_by_ffmpeg(file_path):
    """Checks if a file contains playable audio or video streams using ffprobe."""
    if not os.path.isfile(file_path):
        return False
    if os.path.basename(file_path).startswith('.'):
        return False
    ext = os.path.splitext(file_path)[1].lower()
    if ext and ext not in MEDIA_EXTS:
        return False
    cmd = [
        'ffprobe', '-v', 'error', '-show_entries', 'stream=codec_type',
        '-of', 'default=noprint_wrappers=1:nokey=1', file_path
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=15)
        streams = result.stdout.strip().split('\n')
        return any(s in ('audio', 'video') for s in streams if s)
    except Exception:
        return False

def check_file_settled(file_path, min_age_seconds=15.0, sample_interval=2.0):
    """Verifies that a file is not actively being written or uploaded.

    Returns (is_settled: bool, reason: str).
    """
    if not os.path.isfile(file_path):
        return False, "not a file"
    try:
        now = time.time()
        mtime = os.path.getmtime(file_path)
        age = now - mtime
        if age < min_age_seconds:
            return False, f"modified {age:.1f}s ago (< {min_age_seconds}s threshold)"

        size1 = os.path.getsize(file_path)
        if size1 == 0:
            return False, "file size is 0 bytes (empty/initial handle)"

        time.sleep(sample_interval)
        size2 = os.path.getsize(file_path)
        if size1 != size2:
            return False, f"size changing ({size1:,} -> {size2:,} bytes in {sample_interval}s)"

        return True, "settled"
    except Exception as e:
        return False, str(e)

def promote_settled_incoming(incoming_dir=INCOMING_DIR, target_dir='/content'):
    """Atomically moves fully uploaded/settled media files from incoming_dir to target_dir."""
    if not os.path.exists(incoming_dir):
        return 0, 0

    candidate_files = [
        os.path.join(incoming_dir, f)
        for f in os.listdir(incoming_dir)
        if os.path.isfile(os.path.join(incoming_dir, f))
        and not f.startswith('.')
        and os.path.splitext(f)[1].lower() in MEDIA_EXTS
    ]

    promoted = 0
    in_flight = 0

    for src_path in sorted(candidate_files):
        fname = os.path.basename(src_path)
        settled, reason = check_file_settled(src_path, min_age_seconds=15.0, sample_interval=2.0)
        if settled and is_playable_by_ffmpeg(src_path):
            dst_path = os.path.join(target_dir, fname)
            # Atomic rename on the same filesystem
            os.replace(src_path, dst_path)
            size_mb = os.path.getsize(dst_path) / (1024 * 1024)
            print(f"  [Incoming Promotion] Promoted '{fname}' ({size_mb:.1f} MB) -> {target_dir}")
            promoted += 1
        else:
            in_flight += 1
            print(f"  [Incoming Staging] Holding '{fname}': transfer in-flight ({reason}).")

    return promoted, in_flight

def scan_pending_files(search_dir='/content'):
    """Scan search_dir for media files that need transcription, filtering out in-flight uploads.

    Returns (pending_list: list, in_flight_list: list).
    """
    # 1. Promote any settled files from staging directory
    promote_settled_incoming(INCOMING_DIR, search_dir)

    # 2. Inspect candidate media files in search_dir
    candidate_files = [
        os.path.join(search_dir, f)
        for f in os.listdir(search_dir)
        if os.path.isfile(os.path.join(search_dir, f))
        and not f.startswith('.')
        and not f.endswith('.zip')
        and not f.endswith('.txt')
        and not f.endswith('.json')
        and os.path.splitext(f)[1].lower() in MEDIA_EXTS
    ]

    registry = load_registry()
    pending_list = []
    in_flight_list = []

    for file_path in sorted(candidate_files):
        file_name = os.path.basename(file_path)
        base_name = os.path.splitext(file_name)[0]
        json_path = os.path.join(transcript_dir, f"{base_name}.json")

        # If transcript already exists, it is complete
        if os.path.exists(json_path):
            continue

        # Check if file is settled and playable
        settled, reason = check_file_settled(file_path, min_age_seconds=15.0, sample_interval=1.5)
        if not settled:
            print(f"  [Queue Hold] Skipping '{file_name}': file is actively writing or uploading ({reason}).")
            in_flight_list.append((file_path, reason))
            continue

        if not is_playable_by_ffmpeg(file_path):
            print(f"  [Queue Warning] Skipping '{file_name}': ffprobe could not detect valid audio streams.")
            continue

        file_hash = calculate_sha256(file_path)
        if file_hash in registry:
            del registry[file_hash]
        pending_list.append(file_path)

    save_registry(registry)
    return pending_list, in_flight_list

total_transcribed_this_session = 0
round_num = 1

while True:
    pending, in_flight = scan_pending_files('/content')

    if not pending:
        if in_flight:
            print(f"\n[Queue Waiting] {len(in_flight)} file(s) are actively uploading. Sleeping 15s for transfers to settle...")
            time.sleep(15)
            continue
        if round_num == 1:
            print("No pending files found in /content. All files already have completed JSON transcripts.")
        else:
            print("\nRescan complete: no newly uploaded files detected in /content.")
        break

    print(f"\n{'='*70}")
    print(f"[Queue Round {round_num}] Found {len(pending)} file(s) ready for transcription"
          + (f" ({len(in_flight)} in-flight upload(s) held back)" if in_flight else ""))
    print(f"{'='*70}")

    for idx, file_path in enumerate(pending, 1):
        file_name = os.path.basename(file_path)
        print(f"\n[Round {round_num} | Item {idx}/{len(pending)}] Starting: {file_name}")
        try:
            transcribe_chapter_pipeline(file_path)
            total_transcribed_this_session += 1
        except Exception as e:
            print(f"Failed to transcribe {file_name}: {e}")
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            print("[Memory Status] Cleaned up system and CUDA memory cache for next file.\n")
        print("-" * 50)

    round_num += 1
    print("\nBatch pass completed. Rescanning /content and /content/incoming for any files uploaded during transcription...")

# ─────────────────────────────────────────────────────────────────────────────
# Package and Automatic Download
# ─────────────────────────────────────────────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(transcript_dir, '*.json')))
zip_path = '/content/whisper_json_transcripts.zip'

if not json_files:
    print(f"\nNo completed JSON transcripts found in {transcript_dir} to package.")
else:
    print(f"\n{'='*70}")
    print(f"All transcriptions complete! Packaging {len(json_files)} completed transcript(s)...")
    print(f"{'='*70}")

    if os.path.exists(zip_path):
        try:
            os.remove(zip_path)
        except OSError:
            pass

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        for f in json_files:
            zipf.write(f, os.path.basename(f))

    zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✓ Created {zip_path} ({zip_size_mb:.1f} MB, {len(json_files)} JSON transcripts).")
    print("Initializing browser download automatically...")
    try:
        files.download(zip_path)
    except Exception as e:
        print(f"Notice: Browser download trigger returned: {e}. You can run Step 7 to re-download manually if needed.")


### Step 6: Inspect Saved Transcripts
This code scans and lists all the generated `.json` files in your Google Drive transcripts folder so you can verify them before zipping and downloading.

In [ ]:
import os
import glob
from datetime import datetime

transcript_dir = '/content/drive/MyDrive/Whisper_Transcripts'
json_files = sorted(glob.glob(os.path.join(transcript_dir, '*.json')))

if not json_files:
    print(f"No JSON files found in {transcript_dir}.")
else:
    print(f"Found {len(json_files)} completed JSON transcripts in Drive:\n")
    print(f"{'Filename':<60} | {'Size (MB)':<10} | {'Last Modified':<20}")
    print("-" * 98)
    for f in json_files:
        filename = os.path.basename(f)
        size_mb = round(os.path.getsize(f) / (1024 * 1024), 2)
        mtime = os.path.getmtime(f)
        mtime_str = datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
        print(f"{filename:<60} | {size_mb:<10} | {mtime_str:<20}")

Found 30 completed JSON transcripts in Drive:

Filename                                                     | Size (MB)  | Last Modified       
--------------------------------------------------------------------------------------------------
01 Preface by Gordon W. Allport & Preface to 1984 Edition.json | 0.46       | 2026-10-15 17:40:25 
02 Experiences in a Concentration Camp_1.json                | 1.04       | 2026-10-15 17:41:23 
03 Experiences in a Concentration Camp_2.json                | 1.27       | 2026-10-15 17:42:35 
04 - Robert Greene, 50 Cent  - The 50th Law (Robert Greene, 50 Cent).json | 17.38      | 2026-10-11 04:56:02 
04 Experiences in a Concentration Camp_3.json                | 0.97       | 2026-10-15 17:43:34 
05 Experiences in a Concentration Camp_4.json                | 0.95       | 2026-10-15 17:44:25 
06 Experiences in a Concentration Camp_5.json                | 0.95       | 2026-10-15 17:45:21 
07 - Robert Greene  - The Daily Laws (Robert Greene, Fred Sande

### Step 7: Package and Download All JSON Transcripts
Once you are satisfied with the files listed above, run this cell to bundle them into a single ZIP file and download it to your local machine.

In [ ]:
import zipfile
from google.colab import files

zip_path = '/content/whisper_json_transcripts.zip'

if not json_files:
    print("No files to package.")
else:
    print(f"Packaging {len(json_files)} transcripts into ZIP...")
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for f in json_files:
            zipf.write(f, os.path.basename(f))

    print(f"Successfully created ZIP file at {zip_path}.")
    print("Initializing browser download...")
    files.download(zip_path)

Packaging 30 transcripts into ZIP...
Successfully created ZIP file at /content/whisper_json_transcripts.zip.
Initializing browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>